# H1a SO-101 の体と、指示の出し方

**ねらい**：SO-101 に「関節をこの角度へ」と指示すると、モーターの中で何が起きているかを知る。
**ばね（kp）・減衰（kv）・力の上限** を自分で書き換えて、目標と実際の角度のずれをグラフで確かめる。

この 3 つは、作品の中でずっと効いている。
- **力の上限**：指の力の上限が、そのまま「握る力の設定」（作品の 0.25 N·m など）
- **ばねと減衰**：一気に目標を置くと行き過ぎる（わざと失敗 `jump`）
- **自分の腕の重さ**：肩のモーターがどれだけ力を出しているかで、持った箱の重さを測る（H1c）

**進め方**：上から ▶。「✏️ 変えてみる」のセルは、**予想を 1 行書いてから** 値を変えて ▶ し直す。

## 1. 準備（Colab では 3〜5 分。2 回目からは速い）

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. SO-101 を 1 台置く

`gs.morphs.MJCF`：ロボットの形と関節を書いたファイル（MJCF。MuJoCo の形式）を読む。
`decompose_robot_error_threshold=0.0`：ぶつかり判定のために、形を細かい凸の部品に分けて包む（0 がいちばん細かい。H2 で粗くして失敗させる）。
組み立て（`build`）は時間がかかるので **1 回だけ**。条件を変えるたびに `reset` で初めの状態に戻す。

In [ ]:
gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10), show_viewer=False)
sc.add_entity(gs.morphs.Plane())
arm = sc.add_entity(gs.morphs.MJCF(file=MODEL, decompose_robot_error_threshold=0.0))
cam = sc.add_camera(res=(768, 432), pos=(0.45, -0.45, 0.30), lookat=(0.05, 0.0, 0.12), fov=45, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")

## 3. 関節を一覧にする

SO-101 の関節は 6 つ。5 つが腕、最後の 1 つが **指**（動く側の指を回す関節）。
どの関節にも、角度の範囲と、モーターが出せる **力の上限**（回す力＝トルク、単位 N·m）がある。

In [ ]:
names = [j.name for j in arm.joints if j.n_dofs == 1]
lo, hi = [x.cpu().numpy() for x in arm.get_dofs_limit()]
f_lo, f_hi = [x.cpu().numpy() for x in arm.get_dofs_force_range()]
kp, kv = arm.get_dofs_kp().cpu().numpy(), arm.get_dofs_kv().cpu().numpy()
print(f"{'#':2s} {'関節':14s} {'角度の範囲 [rad]':>18s} {'力の上限 [N·m]':>14s} {'kp':>8s} {'kv':>6s}")
for i, n in enumerate(names):
    print(f"{i:2d} {n:14s} {lo[i]:8.2f} 〜 {hi[i]:5.2f} {f_hi[i]:14.2f} {kp[i]:8.1f} {kv[i]:6.2f}")

**期待される出力**（手元の実測）：6 行。`gripper`（指）は 5 番。力の上限は全部 3.35 N·m、kp は 998.22、kv は 2.73。
kp・kv・力の上限は、実機のモーター（Feetech STS3215）に合わせた値で、モデルのファイルに書いてある。

## 4. 関節に角度を指示して、目標と実際を比べる

`control_dofs_position(目標の角度)`：**「この角度へ行け」という指示**。モーターは、目標とのずれに比例した力（ばね kp）と、速さに比例したブレーキ（減衰 kv）を出す。
出せる力は、力の上限で頭打ちになる。式で書くと：

  力 ＝ kp ×（目標 − 角度） − kv × 角速度　　（ただし ±力の上限 で頭打ち）

`step_response` は、肩（`shoulder_lift`、1 番）だけに 0 → −0.6 rad（後ろへ起こす向き）を指示して、2 秒間の角度と力を記録する関数。
前へ倒す向き（＋）にすると、手先が床に当たって止まり、kp を変えても結果が変わらない（教材を作るときに実際に踏んだ。グラフだけ見ると、うまくいった実験に見える）。

In [ ]:
J = 1   # 肩（shoulder_lift）

def step_response(target=-0.6, kp=998.22, kv=2.731, f_max=3.35, ramp=0, steps=200, mode="position"):
    """肩だけを動かす。ramp > 0 なら、目標を ramp step かけて少しずつ置く（S 字）。記録した角度と力を返す。"""
    sc.reset()
    arm.set_dofs_kp(np.full(6, 998.22)); arm.set_dofs_kv(np.full(6, 2.731))
    arm.set_dofs_force_range(np.full(6, -3.35), np.full(6, 3.35))
    arm.set_dofs_kp([kp], [J]); arm.set_dofs_kv([kv], [J])
    arm.set_dofs_force_range([-f_max], [f_max], [J])
    q, f = [], []
    for k in range(steps):
        s = 1.0 if ramp == 0 else min(1.0, (k + 1) / ramp)
        s = s if ramp == 0 else 3 * s ** 2 - 2 * s ** 3
        goal = np.zeros(6); goal[J] = target * s
        if mode == "position":
            arm.control_dofs_position(goal)
        elif mode == "force":       # 力を直接指示する（ここでは 0 N·m＝力を抜く）
            arm.control_dofs_force(np.zeros(6))
        sc.step()
        q.append(float(arm.get_dofs_position()[J])); f.append(float(arm.get_dofs_control_force()[J]))
    return np.array(q), np.array(f)

def plot(runs, target=-0.6):
    t = np.arange(len(next(iter(runs.values()))[0])) * 0.01
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    for label, (q, f) in runs.items():
        ax[0].plot(t, q, label=label); ax[1].plot(t, f, label=label)
    ax[0].axhline(target, color="gray", ls="--", lw=0.8); ax[0].set(xlabel="s", ylabel="肩の角度 [rad]")
    ax[1].set(xlabel="s", ylabel="肩のモーターの力 [N·m]"); ax[0].legend(); plt.show()
    for label, (q, f) in runs.items():
        u = np.sign(target) * q                       # 目標の向きに測った角度（向きを見ないと、反対へ倒れたのを「届いた」と数える）
        reach = f"{np.argmax(u >= 0.9 * abs(target)) * 0.01:.2f} s" if (u >= 0.9 * abs(target)).any() else "届かない"
        print(f"{label:28s} 行き過ぎ {max(u.max() - abs(target), 0):.3f} rad   9 割に届く {reach:>8s}   2 秒後のずれ {target - q[-1]:+.4f} rad   最後の力 {f[-1]:+.2f} N·m")

plot({"いまの値（kp 998, kv 2.7）": step_response()})

**期待される出力**（手元の実測）
```
いまの値（kp 998, kv 2.7）  行き過ぎ 0.078 rad   9 割に届く   0.18 s   2 秒後のずれ -0.0002 rad   最後の力 -0.21 N·m
```
- 0.18 秒で目標の 9 割に届き、0.078 rad（約 4.5°）行き過ぎてから戻る
- 右のグラフ：力は最初、力の上限（3.35 N·m）に張り付く。止まると −0.21 N·m だけ出し続ける。**これが腕の重さを支える力**（H1c で、箱を持ったときのこの力の増え方から重さを測る）

## ✏️ 変えてみる 1：ばね kp
**予想**：kp を 1/10（100）にすると、行き過ぎと 2 秒後のずれはどうなる？ 10 倍（10000）では？

In [ ]:
plot({f"kp {kp}": step_response(kp=kp) for kp in [20, 100, 998.22, 10000]})

## ✏️ 変えてみる 2：減衰 kv
**予想**：kv を 0 にすると？ 30 にすると？

In [ ]:
plot({f"kv {kv}": step_response(kv=kv) for kv in [0.0, 2.731, 30.0]})

## ✏️ 変えてみる 3：力の上限
**予想**：肩の力の上限を 1.0 N·m、0.3 N·m にすると、腕は目標に届く？
（作品では、**指** の力の上限を 0.2〜0.8 N·m に絞って「握る力」にしている。肩で同じことをすると何が起きるか）

In [ ]:
plot({f"力の上限 {fm} N·m": step_response(f_max=fm) for fm in [3.35, 1.0, 0.3]})

## ✏️ 変えてみる 4：目標の出し方（一気に／S 字で少しずつ）
**予想**：目標を一気に置く（いまの書き方）のと、1 秒かけて S 字で少しずつ置くのとで、行き過ぎはどう違う？
作品の共通部品 `move` は S 字で置いている。一気に置くのが、わざと失敗 `jump`。

In [ ]:
plot({"一気に": step_response(ramp=0), "S 字で 0.5 秒": step_response(ramp=50), "S 字で 1 秒": step_response(ramp=100)})

## ✏️ 変えてみる 5：指示の種類
位置ではなく **力** を指示する（`control_dofs_force`）。0 N·m を指示すると、モーターは力を抜く。
**予想**：肩はどうなる？

In [ ]:
plot({"位置で指示": step_response(), "力 0 を指示": step_response(mode="force")})

## 5. 映像で確かめる
どれか 1 つの条件で、腕の動きを録画して見る。`step_response` の中身と同じ指示を出しながら、4 step に 1 コマ描く。
**変えてみる**：`kp=` や `f_max=` を変えて ▶ し直す。

In [ ]:
from IPython.display import HTML
import base64, imageio

def film(target=-0.6, kp=998.22, kv=2.731, f_max=3.35, steps=150, name="h1a.mp4"):
    sc.reset()
    arm.set_dofs_kp([kp], [J]); arm.set_dofs_kv([kv], [J]); arm.set_dofs_force_range([-f_max], [f_max], [J])
    frames = []
    for k in range(steps):
        goal = np.zeros(6); goal[J] = target
        arm.control_dofs_position(goal); sc.step()
        if k % 4 == 0:
            frames.append(cam.render(rgb=True)[0])
    imageio.mimsave(name, frames, fps=25)
    return HTML(f'<video controls width=640 src="data:video/mp4;base64,{base64.b64encode(open(name, "rb").read()).decode()}">')

film(f_max=0.3)

## 確かめる問い
1. 2 秒後に目標とのずれが残るのはなぜ？ kp を大きくすると小さくなるのはなぜ？
2. 作品で「握る力」を決めているのは、指のモーターの何？
3. 作品の共通部品 `move` が、目標を S 字で少しずつ置くのはなぜ？

<details><summary>答え</summary>

1. 腕の重さを支える力が要るから。ばねの力 kp ×（ずれ）が重さと釣り合う所で止まる。kp が大きいほど、同じ力を小さいずれで出せる
2. 力の上限（`set_dofs_force_range`）。指は「閉じきれ」と指示されていて、箱に当たって止まると、上限いっぱいの力で押す
3. 一気に置くと、ずれが大きく、力の上限いっぱいで加速して行き過ぎるから（わざと失敗 `jump` で箱の上に着地する）
</details>